# Team AEC Possession Shapes

Find a team's highest `aec_per_throw` scoring possessions and compare them with the middle of the filtered efficient-possession set.

In [9]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from ufa_aec_possessions import (
    build_aec_possession_sets,
    build_scoring_possessions,
    fetch_shownspace_season_throws,
    plot_possession_path,
    plot_representative_paths,
)

In [ ]:
TEAM_ID = 'empire'
SEASON = 2026
MAX_GAMES = None
METRIC = 'aec_per_throw'
TOP_N = 5
MIDDLE_COUNT = 5

## Load Shown Space Throws

In [3]:
games, throws = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
)
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f'Games loaded: {len(games):,}')
print(f'Throws loaded: {len(throws):,}')
print(f'Scoring possessions found: {len(possessions):,}')

Games loaded: 10
Throws loaded: 5,550
Scoring possessions found: 215


## Build Highest and Middle AEC Sets

Defaults: O-line scoring possessions, long-field starts, hucks excluded, ranked by `aec_per_throw`.

In [4]:
sets = build_aec_possession_sets(
    possessions,
    paths,
    top_n=TOP_N,
    middle_count=MIDDLE_COUNT,
    metric=METRIC,
)
filtered_possessions, filtered_paths = sets['filtered']
highest_possessions, highest_paths = sets['highest']
middle_possessions, middle_paths = sets['middle']

print(f'Filtered analysis possessions: {len(filtered_possessions):,}')

Filtered analysis possessions: 67


In [5]:
summary_columns = [
    'possession_id', 'GameID', 'line_type', 'start_y', 'end_y',
    'field_progress', 'throw_count', 'huck_count', 'total_aec', METRIC,
    'shape_width', 'shape_middle_third_share', 'shape_sideline_share',
]
highest_possessions.reindex(columns=summary_columns)

,possession_id,GameID,line_type,start_y,end_y,field_progress,throw_count,huck_count,total_aec,aec_per_throw,shape_width,shape_middle_third_share,shape_sideline_share
0,2026-05-31-BOS-TOR|2|8|3|False,2026-05-31-BOS-TOR,o_line,40.45,107.80,67.35,5,0,1.106622,0.221324,17.02,0.400000,0.000000
1,2026-06-05-NY-BOS|3|7|3|True,2026-06-05-NY-BOS,o_line,41.57,113.56,71.99,5,0,1.008691,0.201738,33.67,0.300000,0.200000
2,2026-06-05-NY-BOS|1|9|1|True,2026-06-05-NY-BOS,o_line,40.00,101.62,61.62,9,0,1.177931,0.130881,29.91,0.611111,0.055556
3,2026-05-02-MTL-BOS|3|6|1|True,2026-05-02-MTL-BOS,o_line,33.93,101.61,67.68,8,0,1.002380,0.125297,22.37,0.500000,0.000000
4,2026-06-12-BOS-NY|3|6|1|False,2026-06-12-BOS-NY,o_line,8.43,106.51,98.08,9,0,1.000934,0.111215,33.20,0.555556,0.055556


In [6]:
middle_possessions.reindex(columns=summary_columns)

,possession_id,GameID,line_type,start_y,end_y,field_progress,throw_count,huck_count,total_aec,aec_per_throw,shape_width,shape_middle_third_share,shape_sideline_share
0,2026-04-25-DC-BOS|2|1|1|True,2026-04-25-DC-BOS,o_line,8.45,105.29,96.84,15,0,1.041824,0.069455,29.41,0.466667,0.166667
1,2026-06-05-NY-BOS|1|7|1|True,2026-06-05-NY-BOS,o_line,5.91,103.27,97.36,13,0,0.909633,0.069972,32.64,0.423077,0.076923
2,2026-05-16-BOS-MTL|2|8|1|False,2026-05-16-BOS-MTL,o_line,14.06,111.03,96.97,13,0,0.921364,0.070874,37.14,0.461538,0.153846
3,2026-05-02-MTL-BOS|1|6|1|True,2026-05-02-MTL-BOS,o_line,9.03,102.77,93.74,14,0,1.000000,0.071429,31.99,0.571429,0.035714
4,2026-06-06-BOS-PHI|3|5|1|False,2026-06-06-BOS-PHI,o_line,9.51,102.24,92.73,14,0,1.001209,0.071515,38.35,0.535714,0.071429


## Diagram One Highest Possession

In [7]:
if highest_paths:
    fig = plot_possession_path(
        highest_paths[0],
        title=f'{TEAM_ID.title()} highest non-huck long-field aEC/T possession',
    )
    fig.show()

## Overlay Middle Possessions

In [8]:
middle_lookup = {path['possession_id'].iloc[0]: path for path in middle_paths}
middle_labeled_paths = {
    f'middle {rank + 1}: {row[METRIC]:.3f}': middle_lookup[row['possession_id']]
    for rank, (_, row) in enumerate(middle_possessions.iterrows())
    if row['possession_id'] in middle_lookup
}

if middle_labeled_paths:
    fig = plot_representative_paths(
        middle_labeled_paths,
        title=f'{TEAM_ID.title()} middle {len(middle_labeled_paths)} non-huck long-field scoring possessions',
    )
    fig.show()